%md-sandbox
<div style="background:linear-gradient(135deg,#2f80d7,#173b64);padding:28px;border-radius:16px;color:white">
  <h1 style="margin:0;color:white">Reto — Asistente de E-commerce</h1>
  <h2 style="margin:8px 0;color:white">00 — Datos sintéticos</h2>
  <p style="font-size:16px;margin:14px 0 0 0;color:#eef6ff">
    Genera desde cero, con Faker y seed fija, los datos estructurados del caso:
    clientes, catálogo de productos, pedidos, el detalle de cada pedido y reviews
    de producto.
  </p>
</div>

Este notebook es el único que se entrega armado — el resto del reto (medallion,
RAG, tools, agente, evaluación, endpoint, app) lo construyen ustedes, tomando como
referencia `../../notebooks/` del Proyecto Integrador de RRHH. Ver el `README.md`
de esta carpeta para el enunciado completo.

Escribe cada DataFrame directo como tabla bronze en Unity Catalog — sin landing ni
CSVs de por medio (mismo patrón que `00_Setup_DatosSinteticos.ipynb` del proyecto
de referencia, ya validado en un workspace real).

In [ ]:
try:
    dbutils.widgets.text("CATALOG", "workspace", "Catalog")
    CATALOG = dbutils.widgets.get("CATALOG").strip()
except NameError:
    CATALOG = "workspace"

try:
    dbutils.widgets.text("SCHEMA", "reto_ecommerce", "Schema")
    SCHEMA = dbutils.widgets.get("SCHEMA").strip()
except NameError:
    SCHEMA = "reto_ecommerce"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
print("Catalog :", CATALOG)
print("Schema  :", SCHEMA)

In [ ]:
%pip install faker --quiet

In [ ]:
dbutils.library.restartPython()

In [ ]:
# Reconstruir los widgets/paths después del restart de Python.
try:
    CATALOG = dbutils.widgets.get("CATALOG").strip()
    SCHEMA = dbutils.widgets.get("SCHEMA").strip()
except NameError:
    CATALOG, SCHEMA = "workspace", "reto_ecommerce"

import random
from datetime import date, timedelta

import pandas as pd
from faker import Faker

SEED = 42
random.seed(SEED)
fake = Faker("es_ES")
Faker.seed(SEED)

# N_CLIENTES controla la escala de TODO el dataset -- pedidos, detalle y reviews se
# generan a partir de clientes y productos, así que subir este número alcanza para
# tener bastante más data en las 5 tablas.
N_CLIENTES = 150
N_PRODUCTOS = 80
CLIENTE_IDS = list(range(1, N_CLIENTES + 1))
PRODUCTO_IDS = list(range(1, N_PRODUCTOS + 1))

FECHA_INICIO = date(2024, 1, 1)
FECHA_FIN = date(2025, 12, 31)


def fecha_aleatoria(inicio: date = FECHA_INICIO, fin: date = FECHA_FIN) -> date:
    dias = (fin - inicio).days
    return inicio + timedelta(days=random.randint(0, dias))

## 0. `clientes` — tabla base

In [ ]:
CIUDADES = ["Lima", "Arequipa", "Trujillo", "Cusco", "Piura", "Chiclayo"]
SEGMENTOS = ["Nuevo", "Recurrente", "VIP"]
PESOS_SEGMENTO = [0.5, 0.35, 0.15]

clientes_data = []
for cliente_id in CLIENTE_IDS:
    nombre = fake.name()
    clientes_data.append(
        (
            cliente_id,
            nombre,
            fake.unique.email(),
            random.choice(CIUDADES),
            fecha_aleatoria(date(2022, 1, 1), date(2025, 6, 1)),
            random.choices(SEGMENTOS, weights=PESOS_SEGMENTO)[0],
        )
    )

clientes_df = pd.DataFrame(
    clientes_data,
    columns=["cliente_id", "nombre", "email", "ciudad", "fecha_registro", "segmento"],
)
clientes_df["fecha_registro"] = clientes_df["fecha_registro"].astype(str)
print(len(clientes_df), "clientes")
clientes_df.head()

## 1. `productos` — catálogo

In [ ]:
CATEGORIAS_PRODUCTOS = {
    "Electrónica": (80, 3500),
    "Hogar": (20, 800),
    "Ropa": (15, 350),
    "Deportes": (25, 600),
    "Libros": (10, 120),
    "Juguetes": (15, 300),
}

productos_data = []
for producto_id in PRODUCTO_IDS:
    categoria = random.choice(list(CATEGORIAS_PRODUCTOS.keys()))
    precio_min, precio_max = CATEGORIAS_PRODUCTOS[categoria]
    productos_data.append(
        (
            producto_id,
            f"{categoria} {fake.word().capitalize()} {producto_id}",
            categoria,
            round(random.uniform(precio_min, precio_max), 2),
            random.randint(0, 200),
        )
    )

productos_df = pd.DataFrame(
    productos_data, columns=["producto_id", "nombre", "categoria", "precio", "stock"]
)
print(len(productos_df), "productos")
productos_df.head()

## 2. `pedidos`

In [ ]:
ESTADOS_PEDIDO = ["pendiente", "enviado", "entregado", "cancelado"]
PESOS_ESTADO_PEDIDO = [0.10, 0.15, 0.65, 0.10]

N_PEDIDOS = N_CLIENTES * 6  # ~6 pedidos por cliente en promedio

pedidos_data = []
for pedido_id in range(1, N_PEDIDOS + 1):
    pedidos_data.append(
        (
            pedido_id,
            random.choice(CLIENTE_IDS),
            fecha_aleatoria(),
            random.choices(ESTADOS_PEDIDO, weights=PESOS_ESTADO_PEDIDO)[0],
        )
    )

pedidos_df = pd.DataFrame(pedidos_data, columns=["pedido_id", "cliente_id", "fecha_pedido", "estado"])
pedidos_df["fecha_pedido"] = pedidos_df["fecha_pedido"].astype(str)
print(len(pedidos_df), "pedidos")
pedidos_df.head()

## 3. `detalle_pedidos` — línea a línea, define el `total` de cada pedido

In [ ]:
precio_por_producto = dict(zip(productos_df["producto_id"], productos_df["precio"]))

detalle_data = []
totales_por_pedido = {}
detalle_id = 1
for pedido_id in pedidos_df["pedido_id"]:
    n_items = random.randint(1, 6)
    total_pedido = 0.0
    for producto_id in random.sample(PRODUCTO_IDS, n_items):
        cantidad = random.randint(1, 4)
        precio_unitario = precio_por_producto[producto_id]
        detalle_data.append(
            {
                "detalle_id": detalle_id,
                "pedido_id": pedido_id,
                "producto_id": producto_id,
                "cantidad": cantidad,
                "precio_unitario": precio_unitario,
            }
        )
        total_pedido += cantidad * precio_unitario
        detalle_id += 1
    totales_por_pedido[pedido_id] = round(total_pedido, 2)

detalle_pedidos_df = pd.DataFrame(detalle_data)
pedidos_df["total"] = pedidos_df["pedido_id"].map(totales_por_pedido)
print(len(detalle_pedidos_df), "líneas de detalle")
detalle_pedidos_df.head()

## 4. `reviews` — solo pedidos ya entregados

In [ ]:
COMENTARIOS_POR_PUNTAJE = {
    5: "Excelente producto, superó mis expectativas y llegó a tiempo.",
    4: "Buen producto, cumple con lo prometido.",
    3: "Producto correcto, nada especial.",
    2: "El producto no cumplió del todo lo esperado.",
    1: "Muy decepcionado, no lo recomiendo.",
}

pedidos_entregados = pedidos_df[pedidos_df["estado"] == "entregado"]

reviews_data = []
review_id = 1
for _, pedido in pedidos_entregados.iterrows():
    if random.random() < 0.6:  # no todos los pedidos entregados tienen review
        productos_del_pedido = detalle_pedidos_df.loc[
            detalle_pedidos_df["pedido_id"] == pedido["pedido_id"], "producto_id"
        ]
        if productos_del_pedido.empty:
            continue
        producto_id = random.choice(productos_del_pedido.tolist())
        puntaje = random.choices([1, 2, 3, 4, 5], weights=[0.05, 0.10, 0.20, 0.35, 0.30])[0]
        reviews_data.append(
            {
                "review_id": review_id,
                "producto_id": producto_id,
                "cliente_id": int(pedido["cliente_id"]),
                "puntaje": puntaje,
                "comentario": COMENTARIOS_POR_PUNTAJE[puntaje],
                "fecha": fecha_aleatoria().isoformat(),
            }
        )
        review_id += 1

reviews_df = pd.DataFrame(reviews_data)
print(len(reviews_df), "reviews")
reviews_df.head()

## Escribir directo las tablas de la capa bronze

Sin landing/CSV/Auto Loader de por medio: cada DataFrame se escribe tal cual como
tabla bronze en Unity Catalog. El job de medallion que ustedes armen (bronze -> silver
-> gold) parte de estas 5 tablas.

In [ ]:
tablas_bronze = {
    "bronze_clientes": clientes_df,
    "bronze_productos": productos_df,
    "bronze_pedidos": pedidos_df,
    "bronze_detalle_pedidos": detalle_pedidos_df,
    "bronze_reviews": reviews_df,
}

for nombre_tabla, df in tablas_bronze.items():
    destino = f"{CATALOG}.{SCHEMA}.{nombre_tabla}"
    spark.createDataFrame(df).write.mode("overwrite").saveAsTable(destino)
    print(f"  {nombre_tabla}: {len(df)} filas -> {destino}")

print(f"\nListo. Capa bronze poblada en {CATALOG}.{SCHEMA}")
print("Siguiente paso: arma tu job/flujo de medallion (silver + gold) sobre estas 5 tablas.")